# Lambdas and Method References

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Pass behaviour as a value with a lambda or a method reference, such as a test or a comparator.
- Build comparators with Comparator.comparing, thenComparing and reversed.

**In this notebook**, one idea at a time (run each cell before moving on):
1. A lambda: a function as a value
2. Lambdas in everyday code
3. Method references
4. Comparators: comparing, reversed, thenComparing
5. Passing behaviour to your own methods
6. Variables inside a lambda

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## 1. A lambda: a function as a value

A **lambda** is a small function written as a value: `n -> n * 2` takes `n` and gives `n * 2`. You can store it in a variable and pass it to methods. Its type says what it takes and gives:

| Type | Takes → gives | Call it with |
|---|---|---|
| `Predicate<T>` | a T → true or false | `test(x)` |
| `Function<T, R>` | a T → an R | `apply(x)` |
| `Consumer<T>` | a T → nothing (it does something) | `accept(x)` |

In [ ]:
Predicate<Integer> isEven = n -> n % 2 == 0;
Function<String, Integer> length = s -> s.length();
Consumer<String> shout = s -> IO.println(s.toUpperCase() + "!");

IO.println(isEven.test(4) + " " + isEven.test(7));
IO.println(length.apply("lambda"));
shout.accept("hello");

🧪 **Try it:** write a `Predicate<String>` that tells whether a word starts with `"a"`, and test it on two words.

## 2. Lambdas in everyday code

Many methods take a lambda to know **what to do**: `removeIf` takes the test for the items to remove, `forEach` what to do with each item, and `sort` how to compare two items. A comparator gets two items and returns a negative number, zero or a positive number. `Comparator.reverseOrder()` and `Comparator.comparing(...)` build common ones for you.

In [ ]:
List<Integer> numbers = new ArrayList<>(List.of(5, 12, 3, 8, 21, 6));
numbers.removeIf(n -> n % 3 == 0);
IO.println(numbers);
numbers.sort(Comparator.reverseOrder());
IO.println(numbers);
numbers.forEach(n -> IO.print(n * 10 + " "));
IO.println();

🧪 **Try it:** sort in increasing order with the lambda `(a, b) -> Integer.compare(a, b)`. What does `(a, b) -> Integer.compare(b, a)` do? (Avoid `a - b`: it overflows for numbers far apart.)

## 3. Method references

When a lambda only calls an existing method, you can name the method instead: `s -> s.toUpperCase()` becomes `String::toUpperCase`, and `x -> System.out.println(x)` becomes `System.out::println`. It's the same thing, shorter.

In [ ]:
List<String> words = new ArrayList<>(List.of("pear", "fig", "banana"));
words.replaceAll(String::toUpperCase);       // same as s -> s.toUpperCase()
words.forEach(System.out::println);          // same as w -> System.out.println(w)
words.sort(Comparator.comparing(String::length));
IO.println(words);

🧪 **Try it:** rewrite the two method references as lambdas, run, and check you get the same output.

## 4. Comparators: comparing, reversed, thenComparing

Writing `(a, b) -> …` comparators by hand is error-prone. `Comparator.comparing(Player::name)` (or `comparingInt` for numbers) builds one from the value to compare; `.reversed()` flips it, and `.thenComparing(...)` breaks ties with a second value. The slides build one step by step.

#### 🎞️ Building a comparator

![comparingInt(score) puts lower scores first, reversed() flips it, and thenComparing(name) breaks a tie between Bia and Caio, giving Ana 75, Bia 50, Caio 50](comparator-steps.svg)

In [ ]:
record Player(String name, int score) {}

void main() {
    List<Player> table = new ArrayList<>(List.of(
            new Player("Caio", 50), new Player("Ana", 75), new Player("Bia", 50)));
    table.sort(Comparator.comparing(Player::name));
    IO.println(table);
    table.sort(Comparator.comparingInt(Player::score).reversed().thenComparing(Player::name));
    IO.println(table);
}

🧪 **Try it:** sort by score from the lowest, with ties broken by name in reverse order (`Comparator.comparing(Player::name).reversed()` inside `thenComparing`).

### ✏️ Exercise: Cheapest first

Sort the products from the cheapest; products with the same price go in alphabetical order. Use `Comparator.comparingDouble` and `thenComparing`.

In [ ]:
record Product(String name, double price) {}

/** The names after sortCheapestFirst. */
List<String> names(List<Product> items) {
    List<Product> copy = new ArrayList<>(items);
    sortCheapestFirst(copy);
    return copy.stream().map(Product::name).toList();
}

void sortCheapestFirst(List<Product> items) {
    // TODO: items.sort(Comparator.comparingDouble(Product::price)…)
}

void main() {
    check("by price", names(List.of(new Product("pen", 2.5), new Product("book", 30), new Product("gum", 1))), List.of("gum", "pen", "book"));
    check("a tie", names(List.of(new Product("tea", 4), new Product("coffee", 4))), List.of("coffee", "tea"));
}

<details>
<summary>Show an answer</summary>

```java
record Product(String name, double price) {}

/** The names after sortCheapestFirst. */
List<String> names(List<Product> items) {
    List<Product> copy = new ArrayList<>(items);
    sortCheapestFirst(copy);
    return copy.stream().map(Product::name).toList();
}

void sortCheapestFirst(List<Product> items) {
    items.sort(Comparator.comparingDouble(Product::price).thenComparing(Product::name));
}

void main() {
    check("by price", names(List.of(new Product("pen", 2.5), new Product("book", 30), new Product("gum", 1))), List.of("gum", "pen", "book"));
    check("a tie", names(List.of(new Product("tea", 4), new Product("coffee", 4))), List.of("coffee", "tea"));
}
```

</details>

## 5. Passing behaviour to your own methods

Your methods can take a lambda too: declare a parameter of a functional type, such as `Predicate<String>`, and call its method (`test`) inside. The caller decides the behaviour, and one method serves many cases.

In [ ]:
int countIf(List<String> words, Predicate<String> test) {
    int count = 0;
    for (String w : words) {
        if (test.test(w)) {
            count++;
        }
    }
    return count;
}

void main() {
    List<String> words = List.of("sky", "stream", "lambda", "map");
    IO.println(countIf(words, w -> w.length() <= 3));
    IO.println(countIf(words, w -> w.startsWith("s")));
    IO.println(countIf(words, String::isEmpty));
}

🧪 **Try it:** count the words that contain an "a". Then write `apply(List<String> words, Function<String, String> f)` that returns a new list with `f` applied to each word.

### ✏️ Exercise: Keep the matching words

Return a new list with only the words that pass the test, in order, by calling `test.test(word)`.

In [ ]:
List<String> keep(List<String> words, Predicate<String> test) {
    // TODO: add each word for which test.test(word) is true
    return List.of();
}

void main() {
    check("short words", keep(List.of("a", "bcd", "ef"), w -> w.length() < 3), List.of("a", "ef"));
    check("upper case", keep(List.of("Hi", "hey", "Yo"), w -> Character.isUpperCase(w.charAt(0))), List.of("Hi", "Yo"));
    check("none pass", keep(List.of("x"), w -> false), List.of());
}

<details>
<summary>Show an answer</summary>

```java
List<String> keep(List<String> words, Predicate<String> test) {
    List<String> result = new ArrayList<>();
    for (String w : words) {
        if (test.test(w)) {
            result.add(w);
        }
    }
    return result;
}

void main() {
    check("short words", keep(List.of("a", "bcd", "ef"), w -> w.length() < 3), List.of("a", "ef"));
    check("upper case", keep(List.of("Hi", "hey", "Yo"), w -> Character.isUpperCase(w.charAt(0))), List.of("Hi", "Yo"));
    check("none pass", keep(List.of("x"), w -> false), List.of());
}
```

</details>

## 6. Variables inside a lambda

A lambda can use the local variables around it, but only if they never change after being set ("effectively final"). That stops a lambda from seeing a value that changed after it was created. To accumulate inside a lambda, use a stream (next lesson) or a field instead of a local counter.

In [ ]:
int limit = 3;                                  // never changes: fine
Predicate<String> isShort = w -> w.length() <= limit;
IO.println(isShort.test("map") + " " + isShort.test("stream"));

🧪 **Try it:** add `limit = 4;` after the lambda and read the error ("must be final or effectively final").

## In short

A **lambda** is a small function written as a value: `n -> n % 2 == 0` takes `n` and answers true or false. A **method reference** such as `String::length` names an existing method instead. Many methods take one to know *what to do*: which items to remove, or how to compare two items.

```java
record Player(String name, int score) {}

void main() {
    Predicate<Integer> isEven = n -> n % 2 == 0;
    IO.println(isEven.test(4));

List<Player> table = new ArrayList<>(List.of(
            new Player("Caio", 50), new Player("Ana", 75), new Player("Bia", 50)));
    table.sort(Comparator.comparingInt(Player::score).reversed()
            .thenComparing(Player::name));
    IO.println(table);   // Ana 75, then Bia and Caio, tied at 50
}
```

A comparator returns a negative number, zero or a positive number, saying which of two items comes first.

**Watch out:** a lambda can use a local variable only if that variable never changes after it is set.

## Readings

- [Writing Your First Lambda Expression](https://dev.java/learn/language/fp/lambdas/first-lambdas/) (dev.java (Oracle), 9 min): Why a lambda needs a functional interface, the Predicate and Consumer examples, and how the syntax gets shorter step by step.

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Sort with a Comparator (Lambdas)** · Easy